In [ ]:
ANIMAL DATA INTEGRATION AND CLASSIFICATION 

In [ ]:
import json
import re

# ============================
# 1. LOAD JSON FILE
# ============================
with open("auxiliary_metadata.json", "r") as f:
    metadata = json.load(f)

# Convert JSON to a dict (ensure editable)
metadata = dict(metadata)

# ============================
# 2. STANDARDIZE JSON FIELD NAMES
# ============================

key_map = {
    "conservation": "conservation_status",
    "status": "conservation_status",
    "habitat_type": "habitat",
    "diet_type": "diet"
}

fixed_metadata = {}

for key, value in metadata.items():
    new_key = key_map.get(key, key)   # rename if needed
    fixed_metadata[new_key] = value

# ============================
# 3. FIX TYPOS IN DIET FIELD
# ============================

if "diet" in fixed_metadata:
    diet_value = fixed_metadata["diet"].lower().strip()

    diet_corrections = {
        "omnivor": "omnivore",
        "omnivoer": "omnivore",
        "omnivoure": "omnivore",
        "herbivor": "herbivore",
        "herbavour": "herbivore",
        "carnivor": "carnivore",
        "carnivoer": "carnivore",
    }

    # Replace only if typo exists
    fixed_metadata["diet"] = diet_corrections.get(diet_value, diet_value)

# ============================
# 4. SAVE FIXED JSON BACK
# ============================
with open("cleaned_metadata.json", "w") as f:
    json.dump(fixed_metadata, f, indent=4)

print("Cleaned JSON saved as cleaned_metadata.json")


In [ ]:
# ================================
#  🔥 TASK 1 — LOAD & INTEGRATE DATA
# ================================
import pandas as pd
import json
import re
import matplotlib.pyplot as plt

# Load CSVs
zoo_df = pd.read_csv("zoo.csv")
class_df = pd.read_csv("class.csv")

# Load JSON
with open("auxiliary_metadata.json") as f:
    metadata = json.load(f)

metadata_df = pd.json_normalize(metadata)

# ================================
#  🔥 TASK 2 — CLEANING & NORMALIZATION
# ================================

# --- (A) Normalize names: remove spaces, special chars, lowercase
def clean_name(x):
    return re.sub(r'[^A-Za-z]', '', str(x)).lower()

zoo_df["clean_name"] = zoo_df["name"].apply(clean_name)
class_df["clean_name"] = class_df["name"].apply(clean_name)
metadata_df["clean_name"] = metadata_df["name"].apply(clean_name)

# --- (B) Fix inconsistent column names in JSON
metadata_df.rename(columns={
    "conservation.status": "conservation_status",
    "conservation": "conservation_status",
    "status": "conservation_status",
    "habitat_type": "habitat",
    "habitatType": "habitat",
    "diet_type": "diet",
    "dietType": "diet"
}, inplace=True)

# --- (C) Fix diet typos
metadata_df["diet"] = metadata_df["diet"].replace({
    "omnivor": "omnivore",
    "carvivore": "carnivore",
    "herbavor": "herbivore"
})

# ================================
#  🔥 TASK 3 — MERGING DATASETS
# ================================

# Merge zoo + class (keep ALL zoo rows)
merged_1 = pd.merge(
    zoo_df, class_df,
    on="clean_name",
    how="left"
)

# Merge with JSON (keep all)
merged_final = pd.merge(
    merged_1, metadata_df,
    on="clean_name",
    how="left"
)

# Handle missing metadata
merged_final.fillna({
    "conservation_status": "Unknown",
    "habitat": "Unknown",
    "diet": "Unknown"
}, inplace=True)

# ================================
#  🔥 TASK 4 — SUMMARY & ANALYSIS
# ================================

print("Shape of Final Dataset:", merged_final.shape)
print("\nMissing Values:\n", merged_final.isna().sum())
print("\nDiet Distribution:\n", merged_final["diet"].value_counts())
print("\nConservation Status Distribution:\n", merged_final["conservation_status"].value_counts())

# ================================
#  🔥 PLOT 1 — Diet Distribution
# ================================
merged_final["diet"].value_counts().plot(kind='bar')
plt.title("Diet Distribution")
plt.xlabel("Diet Type")
plt.ylabel("Count")
plt.show()

# ================================
#  🔥 PLOT 2 — Conservation Status
# ================================
merged_final["conservation_status"].value_counts().plot(kind='bar')
plt.title("Conservation Status Distribution")
plt.xlabel("Status")
plt.ylabel("Count")
plt.show()



In [14]:
# Complete notebook-ready pipeline + advanced visualizations
# Copy this entire cell into a Jupyter Notebook and run.

import os
import zipfile
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'figure.max_open_warning': 0})

# ---------------------------
# CONFIG
# ---------------------------
ZIP_NAME = "lab-exam-data.zip"        # change if different
EXTRACT_DIR = "lab_exam_data"         # where ZIP will be extracted
PLOTS_DIR = "plots"
FINAL_CSV = "final_merged_dataset.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

# ---------------------------
# UTILS
# ---------------------------
def find_file_in_dir(dirname, filename):
    for root, _, files in os.walk(dirname):
        if filename in files:
            return os.path.join(root, filename)
    return None

def safe_load_csv(path):
    try:
        df = pd.read_csv(path)
        print(f"Loaded CSV: {path}  (shape: {df.shape})")
        return df
    except Exception as e:
        raise RuntimeError(f"Failed to load CSV {path}: {e}")

def safe_load_json(path):
    try:
        with open(path, 'r') as f:
            data = json.load(f)
        print(f"Loaded JSON: {path}")
        # If metadata is a dict representing a single record => convert to list/dict then df
        if isinstance(data, dict):
            # Try to detect nested structure; flatten to single-row df
            return pd.json_normalize(data)
        elif isinstance(data, list):
            return pd.json_normalize(data)
        else:
            # fallback
            return pd.json_normalize(data)
    except Exception as e:
        raise RuntimeError(f"Failed to load JSON {path}: {e}")

def clean_name(x):
    return re.sub(r'[^A-Za-z]', '', str(x)).lower()

# ---------------------------
# 1) Extract ZIP if exists
# ---------------------------
if os.path.exists(ZIP_NAME):
    print(f"Found zip file {ZIP_NAME}, extracting to {EXTRACT_DIR} ...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_NAME, 'r') as z:
        z.extractall(EXTRACT_DIR)
    base_dir = EXTRACT_DIR
else:
    print(f"No zip named {ZIP_NAME} found in working directory. Looking in current folder for files.")
    base_dir = "."

# ---------------------------
# 2) Locate files
# ---------------------------
zoo_path = find_file_in_dir(base_dir, "zoo.csv")
class_path = find_file_in_dir(base_dir, "class.csv")
meta_path = find_file_in_dir(base_dir, "auxiliary_metadata.json")

print("Detected paths:")
print(" zoo.csv ->", zoo_path)
print(" class.csv ->", class_path)
print(" auxiliary_metadata.json ->", meta_path)

if not zoo_path or not class_path or not meta_path:
    raise FileNotFoundError("One or more required files not found. Ensure zoo.csv, class.csv and auxiliary_metadata.json are present (or properly inside the ZIP).")

# ---------------------------
# 3) Load datasets
# ---------------------------
zoo_df = safe_load_csv(zoo_path)
class_df = safe_load_csv(class_path)
metadata_df = safe_load_json(meta_path)

# ---------------------------
# 4) Basic sanity checks & print columns
# ---------------------------
print("\nColumns found:")
print(" zoo.csv columns:", list(zoo_df.columns))
print(" class.csv columns:", list(class_df.columns))
print(" metadata columns:", list(metadata_df.columns))

# ---------------------------
# 5) Normalize/clean name columns in all datasets
# ---------------------------

# Determine which column likely holds the animal names for each df
# If exact 'name' exists use it, else try to find a column containing 'name' substring.
def pick_name_column(df, default='name'):
    if default in df.columns:
        return default
    for c in df.columns:
        if 'name' in c.lower():
            return c
    # fallback to first object/string column
    for c in df.columns:
        if df[c].dtype == object:
            return c
    raise RuntimeError("Couldn't determine name column for dataframe. Please ensure there's a column with names.")

zoo_name_col = pick_name_column(zoo_df)
class_name_col = pick_name_column(class_df)
meta_name_col = pick_name_column(metadata_df) if len(metadata_df.columns)>0 else None

print("\nUsing name columns:")
print(" zoo name column ->", zoo_name_col)
print(" class name column ->", class_name_col)
print(" metadata name column ->", meta_name_col)

zoo_df['clean_name'] = zoo_df[zoo_name_col].apply(clean_name)
class_df['clean_name'] = class_df[class_name_col].apply(clean_name)
if meta_name_col:
    metadata_df['clean_name'] = metadata_df[meta_name_col].apply(clean_name)
else:
    metadata_df['clean_name'] = None

# ---------------------------
# 6) Standardize metadata (JSON) columns
# ---------------------------
# Normalize possible nested keys (common variants). We'll rename any matching keys.
rename_map = {
    "conservation": "conservation_status",
    "status": "conservation_status",
    "conservation.status": "conservation_status",
    "conservation_status": "conservation_status",
    "habitat_type": "habitat",
    "habitatType": "habitat",
    "habitat": "habitat",
    "diet_type": "diet",
    "dietType": "diet",
    "diet": "diet"
}

# Lowercase column names for robust matching then reapply mapping
lower_map = {c: c for c in metadata_df.columns}
# Complete notebook-ready pipeline + advanced visualizations
# Copy this entire cell into a Jupyter Notebook and run.

import os
import zipfile
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'figure.max_open_warning': 0})

# ---------------------------
# CONFIG
# ---------------------------
ZIP_NAME = "lab-exam-data.zip"        # change if different
EXTRACT_DIR = "lab_exam_data"         # where ZIP will be extracted
PLOTS_DIR = "plots"
FINAL_CSV = "final_merged_dataset.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

# ---------------------------
# UTILS
# ---------------------------
def find_file_in_dir(dirname, filename):
    for root, _, files in os.walk(dirname):
        if filename in files:
            return os.path.join(root, filename)
    return None

def safe_load_csv(path):
    try:
        df = pd.read_csv(path)
        print(f"Loaded CSV: {path}  (shape: {df.shape})")
        return df
    except Exception as e:
        raise RuntimeError(f"Failed to load CSV {path}: {e}")

def safe_load_json(path):
    try:
        with open(path, 'r') as f:
            data = json.load(f)
        print(f"Loaded JSON: {path}")
        # If metadata is a dict representing a single record => convert to list/dict then df
        if isinstance(data, dict):
            # Try to detect nested structure; flatten to single-row df
            return pd.json_normalize(data)
        elif isinstance(data, list):
            return pd.json_normalize(data)
        else:
            # fallback
            return pd.json_normalize(data)
    except Exception as e:
        raise RuntimeError(f"Failed to load JSON {path}: {e}")

def clean_name(x):
    return re.sub(r'[^A-Za-z]', '', str(x)).lower()

# ---------------------------
# 1) Extract ZIP if exists
# ---------------------------
if os.path.exists(ZIP_NAME):
    print(f"Found zip file {ZIP_NAME}, extracting to {EXTRACT_DIR} ...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_NAME, 'r') as z:
        z.extractall(EXTRACT_DIR)
    base_dir = EXTRACT_DIR
else:
    print(f"No zip named {ZIP_NAME} found in working directory. Looking in current folder for files.")
    base_dir = "."

# ---------------------------
# 2) Locate files
# ---------------------------
zoo_path = find_file_in_dir(base_dir, "zoo.csv")
class_path = find_file_in_dir(base_dir, "class.csv")
meta_path = find_file_in_dir(base_dir, "auxiliary_metadata.json")

print("Detected paths:")
print(" zoo.csv ->", zoo_path)
print(" class.csv ->", class_path)
print(" auxiliary_metadata.json ->", meta_path)

if not zoo_path or not class_path or not meta_path:
    raise FileNotFoundError("One or more required files not found. Ensure zoo.csv, class.csv and auxiliary_metadata.json are present (or properly inside the ZIP).")

# ---------------------------
# 3) Load datasets
# ---------------------------
zoo_df = safe_load_csv(zoo_path)
class_df = safe_load_csv(class_path)
metadata_df = safe_load_json(meta_path)

# ---------------------------
# 4) Basic sanity checks & print columns
# ---------------------------
print("\nColumns found:")
print(" zoo.csv columns:", list(zoo_df.columns))
print(" class.csv columns:", list(class_df.columns))
print(" metadata columns:", list(metadata_df.columns))

# ---------------------------
# 5) Normalize/clean name columns in all datasets
# ---------------------------

# Determine which column likely holds the animal names for each df
# If exact 'name' exists use it, else try to find a column containing 'name' substring.
def pick_name_column(df, default='name'):
    if default in df.columns:
        return default
    for c in df.columns:
        if 'name' in c.lower():
            return c
    # fallback to first object/string column
    for c in df.columns:
        if df[c].dtype == object:
            return c
    raise RuntimeError("Couldn't determine name column for dataframe. Please ensure there's a column with names.")

zoo_name_col = pick_name_column(zoo_df)
class_name_col = pick_name_column(class_df)
meta_name_col = pick_name_column(metadata_df) if len(metadata_df.columns)>0 else None

print("\nUsing name columns:")
print(" zoo name column ->", zoo_name_col)
print(" class name column ->", class_name_col)
print(" metadata name column ->", meta_name_col)

zoo_df['clean_name'] = zoo_df[zoo_name_col].apply(clean_name)
class_df['clean_name'] = class_df[class_name_col].apply(clean_name)
if meta_name_col:
    metadata_df['clean_name'] = metadata_df[meta_name_col].apply(clean_name)
else:
    metadata_df['clean_name'] = None

# ---------------------------
# 6) Standardize metadata (JSON) columns
# ---------------------------
# Normalize possible nested keys (common variants). We'll rename any matching keys.
rename_map = {
    "conservation": "conservation_status",
    "status": "conservation_status",
    "conservation.status": "conservation_status",
    "conservation_status": "conservation_status",
    "habitat_type": "habitat",
    "habitatType": "habitat",
    "habitat": "habitat",
    "diet_type": "diet",
    "dietType": "diet",
    "diet": "diet"
}

# Lowercase column names for robust matching then reapply mapping
lower_map = {c: c for c in metadata_df.columns}
lower_cols = {c: c.lower(): c for c in metadata_df.columns}
for lower, orig in lower_cols.items():
    if lower in rename_map:
        metadata_df.rename(columns={orig: rename_map[lower]}, inplace=True)

# Also if keys like 'conservation.status' exist
for c in list(metadata_df.columns):
    c_key = c.replace('.', '_')
    if c_key in rename_map and c != rename_map[c_key]:
        metadata_df.rename(columns={c: rename_map[c_key]}, inplace=True)

# Show metadata columns after rename attempt
print("\nMetadata columns after rename attempt:", list(metadata_df.columns))

# ---------------------------
# 7) Fix diet typos and standardize diet/habitat/conservation values
# ---------------------------
def standardize_text_col(df, col_name):
    if col_name not in df.columns:
        return
    df[col_name] = df[col_name].astype(str).str.strip().str.lower()
    df[col_name] = df[col_name].replace({'nan':'unknown', 'none':'unknown', '': 'unknown'})

# Diet typo dictionary (extendable)
diet_fixes = {
    "omnivor": "omnivore",
    "omnivoer": "omnivore",
    "omnivoure": "omnivore",
    "herbivor": "herbivore",
    "herbavour": "herbivore",
    "herbavor": "herbivore",
    "carnivor": "carnivore",
    "carnivoer": "carnivore",
    "carvivore": "carnivore"
}

if 'diet' in metadata_df.columns:
    metadata_df['diet'] = metadata_df['diet'].astype(str).str.strip().str.lower()
    metadata_df['diet'] = metadata_df['diet'].replace(diet_fixes)
else:
    metadata_df['diet'] = np.nan

# Standardize habitat & conservation columns
standardize_text_col(metadata_df, 'habitat')
standardize_text_col(metadata_df, 'conservation_status')

# If metadata_df has columns like 'conservationStatus' etc attempt to rename them:
for c in list(metadata_df.columns):
    lc = c.lower()
    if 'conserv' in lc and 'conservation_status' not in metadata_df.columns:
        metadata_df.rename(columns={c: 'conservation_status'}, inplace=True)
    if 'habitat' in lc and 'habitat' not in metadata_df.columns:
        metadata_df.rename(columns={c: 'habitat'}, inplace=True)
    if 'diet' in lc and 'diet' not in metadata_df.columns:
        metadata_df.rename(columns={c: 'diet'}, inplace=True)

# After all renames, ensure columns exist
for col in ['conservation_status', 'habitat', 'diet']:
    if col not in metadata_df.columns:
        metadata_df[col] = np.nan

# ---------------------------
# 8) Merge datasets (keep all rows from zoo_df as primary)
# ---------------------------
merged_1 = pd.merge(zoo_df, class_df.drop(columns=[c for c in class_df.columns if c=='clean_name' and c not in ['clean_name']]), on='clean_name', how='left', suffixes=('_zoo','_class'))
# Note: above drop is defensive; ensure we don't duplicate clean_name columns

merged_final = pd.merge(merged_1, metadata_df, on='clean_name', how='left', suffixes=('','_meta'))

# ---------------------------
# 9) Fill missing metadata values with 'Unknown' or proper default
# ---------------------------
merged_final['conservation_status'] = merged_final.get('conservation_status').fillna('unknown')
merged_final['habitat'] = merged_final.get('habitat').fillna('unknown')
merged_final['diet'] = merged_final.get('diet').fillna('unknown')

# If there are multiple name columns, keep original readable ones
# Ensure there is a readable name column called 'animal_name' preferring zoo's original
if 'name_zoo' in merged_final.columns:
    merged_final['animal_name'] = merged_final['name_zoo']
elif 'name' in merged_final.columns:
    merged_final['animal_name'] = merged_final['name']
else:
    # fallback to clean_name capitalized
    merged_final['animal_name'] = merged_final['clean_name'].str.capitalize()

# ---------------------------
# 10) Export final CSV
# ---------------------------
merged_final.to_csv(FINAL_CSV, index=False)
print(f"\nFinal merged dataset saved as: {FINAL_CSV}  (shape: {merged_final.shape})")

# ---------------------------
# 11) VISUALIZATIONS (Advanced)
# ---------------------------

# Helper to save + show
def save_and_show(fig, filename):
    path = os.path.join(PLOTS_DIR, filename)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print("Saved plot:", path)
    plt.show()

sns.set_style("whitegrid")

# 1) Diet Distribution Bar
fig, ax = plt.subplots(figsize=(8,5))
order = merged_final['diet'].value_counts().index
sns.countplot(data=merged_final, x='diet', order=order, ax=ax)
ax.set_title("Diet Distribution")
ax.set_xlabel("Diet Type")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
save_and_show(fig, "diet_distribution.png")

# 2) Conservation Status Distribution
fig, ax = plt.subplots(figsize=(8,5))
order = merged_final['conservation_status'].value_counts().index
sns.countplot(data=merged_final, x='conservation_status', order=order, ax=ax)
ax.set_title("Conservation Status Distribution")
ax.set_xlabel("Conservation Status")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
save_and_show(fig, "conservation_status.png")

# 3) Habitat Distribution
fig, ax = plt.subplots(figsize=(9,5))
order = merged_final['habitat'].value_counts().index
sns.countplot(data=merged_final, x='habitat', order=order, ax=ax)
ax.set_title("Habitat Distribution")
ax.set_xlabel("Habitat Type")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
save_and_show(fig, "habitat_distribution.png")

# 4) Habitat vs Diet Heatmap
try:
    habitat_diet = merged_final.pivot_table(index='habitat', columns='diet', aggfunc='size', fill_value=0)
    fig, ax = plt.subplots(figsize=(10,7))
    sns.heatmap(habitat_diet, annot=True, fmt="d", cmap="YlGnBu", ax=ax)
    ax.set_title("Habitat vs Diet")
    save_and_show(fig, "habitat_vs_diet_heatmap.png")
except Exception as e:
    print("Could not create habitat vs diet heatmap:", e)

# 5) Class vs Diet (countplot) - trying to identify class column variants
possible_class_cols = [c for c in merged_final.columns if 'class' in c.lower()]
class_col = possible_class_cols[0] if possible_class_cols else None
if class_col:
    fig, ax = plt.subplots(figsize=(12,6))
    sns.countplot(data=merged_final, x=class_col, hue='diet', ax=ax)
    ax.set_title(f"Class vs Diet ({class_col})")
    plt.xticks(rotation=45)
    save_and_show(fig, f"class_vs_diet_{class_col}.png")
else:
    print("No class column detected for Class vs Diet plot. Columns looked like:", possible_class_cols)

# 6) Correlation heatmap for numeric features
num_df = merged_final.select_dtypes(include=[np.number])
if not num_df.empty:
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm", ax=ax)
    ax.set_title("Correlation Heatmap (Numeric Features)")
    save_and_show(fig, "correlation_heatmap.png")
else:
    print("No numeric columns found for correlation heatmap.")

# 7) Distribution of 'legs' if present
if 'legs' in merged_final.columns:
    fig, ax = plt.subplots(figsize=(8,5))
    sns.histplot(merged_final['legs'].dropna(), kde=True, ax=ax)
    ax.set_title("Distribution of Number of Legs")
    ax.set_xlabel("Number of Legs")
    save_and_show(fig, "legs_distribution.png")
else:
    print("'legs' column not present, skipping legs distribution.")

# 8) Boxplot of legs by class if both present
if 'legs' in merged_final.columns and class_col:
    fig, ax = plt.subplots(figsize=(12,6))
    sns.boxplot(data=merged_final, x=class_col, y='legs', ax=ax)
    ax.set_title("Legs by Class")
    plt.xticks(rotation=45)
    save_and_show(fig, "legs_by_class_boxplot.png")

# 9) Pairplot of numeric features (small datasets only)
if len(num_df.columns) > 0 and merged_final.shape[0] <= 2000:
    try:
        sns.pairplot(num_df.dropna())
        plt.suptitle("Pairplot (numeric features)", y=1.02)
        plt.show()
    except Exception as e:
        print("Pairplot failed:", e)

print("\nAll requested plots were created (or skipped if not applicable).")
print(f"Plots saved in folder: {os.path.abspath(PLOTS_DIR)}")
print(f"Final merged CSV: {os.path.abspath(FINAL_CSV)}")

for lower, orig in lower_cols.items():
    if lower in rename_map:
        metadata_df.rename(columns={orig: rename_map[lower]}, inplace=True)

# Also if keys like 'conservation.status' exist
for c in list(metadata_df.columns):
    c_key = c.replace('.', '_')
    if c_key in rename_map and c != rename_map[c_key]:
        metadata_df.rename(columns={c: rename_map[c_key]}, inplace=True)

# Show metadata columns after rename attempt
print("\nMetadata columns after rename attempt:", list(metadata_df.columns))

# ---------------------------
# 7) Fix diet typos and standardize diet/habitat/conservation values
# ---------------------------
def standardize_text_col(df, col_name):
    if col_name not in df.columns:
        return
    df[col_name] = df[col_name].astype(str).str.strip().str.lower()
    df[col_name] = df[col_name].replace({'nan':'unknown', 'none':'unknown', '': 'unknown'})

# Diet typo dictionary (extendable)
diet_fixes = {
    "omnivor": "omnivore",
    "omnivoer": "omnivore",
    "omnivoure": "omnivore",
    "herbivor": "herbivore",
    "herbavour": "herbivore",
    "herbavor": "herbivore",
    "carnivor": "carnivore",
    "carnivoer": "carnivore",
    "carvivore": "carnivore"
}

if 'diet' in metadata_df.columns:
    metadata_df['diet'] = metadata_df['diet'].astype(str).str.strip().str.lower()
    metadata_df['diet'] = metadata_df['diet'].replace(diet_fixes)
else:
    metadata_df['diet'] = np.nan

# Standardize habitat & conservation columns
standardize_text_col(metadata_df, 'habitat')
standardize_text_col(metadata_df, 'conservation_status')

# If metadata_df has columns like 'conservationStatus' etc attempt to rename them:
for c in list(metadata_df.columns):
    lc = c.lower()
    if 'conserv' in lc and 'conservation_status' not in metadata_df.columns:
        metadata_df.rename(columns={c: 'conservation_status'}, inplace=True)
    if 'habitat' in lc and 'habitat' not in metadata_df.columns:
        metadata_df.rename(columns={c: 'habitat'}, inplace=True)
    if 'diet' in lc and 'diet' not in metadata_df.columns:
        metadata_df.rename(columns={c: 'diet'}, inplace=True)

# After all renames, ensure columns exist
for col in ['conservation_status', 'habitat', 'diet']:
    if col not in metadata_df.columns:
        metadata_df[col] = np.nan

# ---------------------------
# 8) Merge datasets (keep all rows from zoo_df as primary)
# ---------------------------
merged_1 = pd.merge(zoo_df, class_df.drop(columns=[c for c in class_df.columns if c=='clean_name' and c not in ['clean_name']]), on='clean_name', how='left', suffixes=('_zoo','_class'))
# Note: above drop is defensive; ensure we don't duplicate clean_name columns

merged_final = pd.merge(merged_1, metadata_df, on='clean_name', how='left', suffixes=('','_meta'))

# ---------------------------
# 9) Fill missing metadata values with 'Unknown' or proper default
# ---------------------------
merged_final['conservation_status'] = merged_final.get('conservation_status').fillna('unknown')
merged_final['habitat'] = merged_final.get('habitat').fillna('unknown')
merged_final['diet'] = merged_final.get('diet').fillna('unknown')

# If there are multiple name columns, keep original readable ones
# Ensure there is a readable name column called 'animal_name' preferring zoo's original
if 'name_zoo' in merged_final.columns:
    merged_final['animal_name'] = merged_final['name_zoo']
elif 'name' in merged_final.columns:
    merged_final['animal_name'] = merged_final['name']
else:
    # fallback to clean_name capitalized
    merged_final['animal_name'] = merged_final['clean_name'].str.capitalize()

# ---------------------------
# 10) Export final CSV
# ---------------------------
merged_final.to_csv(FINAL_CSV, index=False)
print(f"\nFinal merged dataset saved as: {FINAL_CSV}  (shape: {merged_final.shape})")

# ---------------------------
# 11) VISUALIZATIONS (Advanced)
# ---------------------------

# Helper to save + show
def save_and_show(fig, filename):
    path = os.path.join(PLOTS_DIR, filename)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print("Saved plot:", path)
    plt.show()

sns.set_style("whitegrid")

# 1) Diet Distribution Bar
fig, ax = plt.subplots(figsize=(8,5))
order = merged_final['diet'].value_counts().index
sns.countplot(data=merged_final, x='diet', order=order, ax=ax)
ax.set_title("Diet Distribution")
ax.set_xlabel("Diet Type")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
save_and_show(fig, "diet_distribution.png")

# 2) Conservation Status Distribution
fig, ax = plt.subplots(figsize=(8,5))
order = merged_final['conservation_status'].value_counts().index
sns.countplot(data=merged_final, x='conservation_status', order=order, ax=ax)
ax.set_title("Conservation Status Distribution")
ax.set_xlabel("Conservation Status")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
save_and_show(fig, "conservation_status.png")

# 3) Habitat Distribution
fig, ax = plt.subplots(figsize=(9,5))
order = merged_final['habitat'].value_counts().index
sns.countplot(data=merged_final, x='habitat', order=order, ax=ax)
ax.set_title("Habitat Distribution")
ax.set_xlabel("Habitat Type")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
save_and_show(fig, "habitat_distribution.png")

# 4) Habitat vs Diet Heatmap
try:
    habitat_diet = merged_final.pivot_table(index='habitat', columns='diet', aggfunc='size', fill_value=0)
    fig, ax = plt.subplots(figsize=(10,7))
    sns.heatmap(habitat_diet, annot=True, fmt="d", cmap="YlGnBu", ax=ax)
    ax.set_title("Habitat vs Diet")
    save_and_show(fig, "habitat_vs_diet_heatmap.png")
except Exception as e:
    print("Could not create habitat vs diet heatmap:", e)

# 5) Class vs Diet (countplot) - trying to identify class column variants
possible_class_cols = [c for c in merged_final.columns if 'class' in c.lower()]
class_col = possible_class_cols[0] if possible_class_cols else None
if class_col:
    fig, ax = plt.subplots(figsize=(12,6))
    sns.countplot(data=merged_final, x=class_col, hue='diet', ax=ax)
    ax.set_title(f"Class vs Diet ({class_col})")
    plt.xticks(rotation=45)
    save_and_show(fig, f"class_vs_diet_{class_col}.png")
else:
    print("No class column detected for Class vs Diet plot. Columns looked like:", possible_class_cols)

# 6) Correlation heatmap for numeric features
num_df = merged_final.select_dtypes(include=[np.number])
if not num_df.empty:
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm", ax=ax)
    ax.set_title("Correlation Heatmap (Numeric Features)")
    save_and_show(fig, "correlation_heatmap.png")
else:
    print("No numeric columns found for correlation heatmap.")

# 7) Distribution of 'legs' if present
if 'legs' in merged_final.columns:
    fig, ax = plt.subplots(figsize=(8,5))
    sns.histplot(merged_final['legs'].dropna(), kde=True, ax=ax)
    ax.set_title("Distribution of Number of Legs")
    ax.set_xlabel("Number of Legs")
    save_and_show(fig, "legs_distribution.png")
else:
    print("'legs' column not present, skipping legs distribution.")

# 8) Boxplot of legs by class if both present
if 'legs' in merged_final.columns and class_col:
    fig, ax = plt.subplots(figsize=(12,6))
    sns.boxplot(data=merged_final, x=class_col, y='legs', ax=ax)
    ax.set_title("Legs by Class")
    plt.xticks(rotation=45)
    save_and_show(fig, "legs_by_class_boxplot.png")

# 9) Pairplot of numeric features (small datasets only)
if len(num_df.columns) > 0 and merged_final.shape[0] <= 2000:
    try:
        sns.pairplot(num_df.dropna())
        plt.suptitle("Pairplot (numeric features)", y=1.02)
        plt.show()
    except Exception as e:
        print("Pairplot failed:", e)

print("\nAll requested plots were created (or skipped if not applicable).")
print(f"Plots saved in folder: {os.path.abspath(PLOTS_DIR)}")
print(f"Final merged CSV: {os.path.abspath(FINAL_CSV)}")


SyntaxError: invalid syntax (81847711.py, line 317)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
class_percent = merged_final["class_type"].value_counts(normalize=True) * 100
class_percent.plot(kind="barh", edgecolor="black")
plt.title("Class Distribution (%)")
plt.xlabel("Percentage")
plt.ylabel("Class Type")
plt.show()


In [ ]:
import seaborn as sns
plt.figure(figsize=(10,6))
sns.boxplot(data=merged_final, x="class_type", y="diet_complexity")
plt.title("Diet Complexity vs Animal Class")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(10,6))
sns.boxplot(data=merged_final, x="class_type", y="is_endangered")
plt.title("Endangerment (Binary) vs Animal Class")
plt.xticks(rotation=45)
plt.show()


In [ ]:
import seaborn as sns

sns.pairplot(
    merged_final,
    vars=["legs", "is_endangered", "diet_complexity"],
    hue="diet"
)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.clustermap(
    merged_final[["legs", "is_endangered", "diet_complexity"]].corr(),
    annot=True,
    cmap="coolwarm"
)
plt.show()


In [ ]:
# =============================
# 🔥 TASK — FEATURE ENGINEERING
# =============================

# --- (A) Create is_endangered binary feature
endangered_list = ["vulnerable", "endangered", "critically endangered"]

merged_final["is_endangered"] = merged_final["conservation_status"].str.lower().isin(endangered_list).astype(int)

# --- (B) Create diet_complexity feature
diet_map = {
    "carnivore": 3,
    "omnivore": 2,
    "herbivore": 1,
    "unknown": 0,
    "other": 0
}

merged_final["diet_complexity"] = merged_final["diet"].str.lower().map(diet_map).fillna(0)

print("✔ Feature Engineering Completed!")
merged_final[["name", "conservation_status", "is_endangered", "diet", "diet_complexity"]].head()


| name  | conservation_status | is_endangered | diet      | diet_complexity |
| ----- | ------------------- | ------------- | --------- | --------------- |
| lion  | vulnerable          | 1             | carnivore | 3               |
| zebra | least concern       | 0             | herbivore | 1               |
| bear  | endangered          | 1             | omnivore  | 2               |
| frog  | unknown             | 0             | unknown   | 0               |
